In [ ]:
!pip install datasets


In [ ]:
!pip install --upgrade datasets transformers fsspec

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 12.5 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
  Attempting uninstall: datasets
    Found existing installation: datasets 2.14.4
    Uninstalling datasets-2.14.4:
      Successfully uninstalled datasets-2.14.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2025.3.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; pla

In [ ]:
from datasets import load_dataset, DatasetDict # Import DatasetDict for explicit use
from transformers import AutoTokenizer

class SummarizationDatasetLoader:
    def __init__(self, dataset_name='cnn_dailymail', dataset_config=None, tokenizer_name='facebook/bart-base', max_input_len=1024, max_target_len=128, sample_size=None):
        self.dataset_name = dataset_name
        self.dataset_config = dataset_config # To handle configs like '3.0.0' for cnn_dailymail or 'arxiv' for scientific_papers
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
        self.max_input_len = max_input_len
        self.max_target_len = max_target_len
        self.sample_size = sample_size

    def _load_raw_dataset(self): # Renamed to avoid clash
        if self.dataset_name == 'cnn_dailymail':
            if not self.dataset_config: self.dataset_config = '3.0.0' # Default if not specified
            dataset = load_dataset('cnn_dailymail', self.dataset_config)
        elif self.dataset_name == 'xsum':
            dataset = load_dataset('xsum')
        elif self.dataset_name == 'billsum':
            dataset = load_dataset('billsum')
        elif self.dataset_name == 'scientific_papers':
            if not self.dataset_config: self.dataset_config = 'arxiv' # Default if not specified
            dataset = load_dataset('scientific_papers', self.dataset_config)
        else:
            raise ValueError(f"Unsupported dataset: {self.dataset_name}")

        if self.sample_size:
            sampled_dataset_dict = DatasetDict()
            for split_name, split_data in dataset.items():
                if len(split_data) > self.sample_size:
                    print(f"Sampling {self.sample_size} examples from {split_name} split...")
                    sampled_dataset_dict[split_name] = split_data.shuffle(seed=42).select(range(self.sample_size))
                else:
                    print(f"Using full {split_name} split ({len(split_data)} examples) as it's smaller than sample_size.")
                    sampled_dataset_dict[split_name] = split_data
            dataset = sampled_dataset_dict

        return dataset

    def preprocess_function(self, examples, input_key, summary_key): # input_key, summary_key passed from map
        inputs = examples[input_key]
        targets = examples[summary_key]

        model_inputs = self.tokenizer(inputs, max_length=self.max_input_len, truncation=True, padding='max_length')

        # Setup tokenizer for targets
        with self.tokenizer.as_target_tokenizer():
            labels = self.tokenizer(targets, max_length=self.max_target_len, truncation=True, padding='max_length')

        # Replace tokenizer.pad_token_id in labels with -100 for loss calculation
        model_inputs['labels'] = []
        for i in range(len(labels['input_ids'])):
            if self.tokenizer.pad_token_id is not None:
                model_inputs['labels'].append([
                    (l if l != self.tokenizer.pad_token_id else -100) for l in labels['input_ids'][i]
                ])
            else: # Fallback if pad_token_id is None, though rare for seq2seq models
                model_inputs['labels'].append(labels['input_ids'][i])

        return model_inputs

    def get_preprocessed_dataset(self):
        dataset = self._load_raw_dataset() # Call the renamed method

        # Determine correct keys based on dataset name
        input_key, summary_key = None, None
        if self.dataset_name == 'cnn_dailymail':
            input_key, summary_key = 'article', 'highlights'
        elif self.dataset_name == 'xsum':
            input_key, summary_key = 'document', 'summary'
        elif self.dataset_name == 'billsum':
            input_key, summary_key = 'text', 'summary'
        elif self.dataset_name == 'scientific_papers':
            input_key, summary_key = 'article', 'abstract'
        else:
            # This case should ideally not be reached if _load_raw_dataset validates dataset_name
            raise ValueError(f"Unknown dataset for key mapping: {self.dataset_name}")

        # Apply preprocessing across all splits in the DatasetDict
        tokenized_dataset = dataset.map(
            lambda examples: self.preprocess_function(examples, input_key, summary_key),
            batched=True,
            remove_columns=dataset['train'].column_names # Remove original columns to save memory and ensure correct format
        )

        return tokenized_dataset

# Example usage
if __name__ == "__main__":
    print("--- Testing CNN/DailyMail Loader ---")
    cnn_loader = SummarizationDatasetLoader(dataset_name='cnn_dailymail', dataset_config='3.0.0', sample_size=100)
    cnn_data = cnn_loader.get_preprocessed_dataset()
    print(f"Loaded CNN/DailyMail splits: {cnn_data.keys()}")
    if 'train' in cnn_data:
        print(f"First train example from CNN/DailyMail:\n{cnn_data['train'][0]}")
        print(f"Input IDs length: {len(cnn_data['train'][0]['input_ids'])}")
        print(f"Labels length: {len(cnn_data['train'][0]['labels'])}")
        # Verify that padding in labels is -100
        print(f"Labels padding check (should be -100): {cnn_data['train'][0]['labels'][-5:]}")


    print("\n--- Testing XSum Loader ---")
    xsum_loader = SummarizationDatasetLoader(dataset_name='xsum', sample_size=50)
    xsum_data = xsum_loader.get_preprocessed_dataset()
    print(f"Loaded XSum splits: {xsum_data.keys()}")
    if 'train' in xsum_data:
        print(f"First train example from XSum:\n{xsum_data['train'][0]}")

    print("\n--- Testing Scientific Papers (Arxiv) Loader ---")
    arxiv_loader = SummarizationDatasetLoader(dataset_name='scientific_papers', dataset_config='arxiv', sample_size=10)
    arxiv_data = arxiv_loader.get_preprocessed_dataset()
    print(f"Loaded Scientific Papers (Arxiv) splits: {arxiv_data.keys()}")
    if 'train' in arxiv_data:
        print(f"First train example from Scientific Papers (Arxiv):\n{arxiv_data['train'][0]}")

--- Testing CNN/DailyMail Loader ---


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/15.6k [00:00<?, ?B/s]

train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

...

Sampling 10 examples from train split...
Sampling 10 examples from validation split...
Sampling 10 examples from test split...


Map:   0%|          | 0/10 [00:00<?, ? examples/s]

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

Loaded Scientific Papers (Arxiv) splits: dict_keys(['train', 'validation', 'test'])
First train example from Scientific Papers (Arxiv):
{'input_ids': [0, 11230, 1437, 14576, 16, 5, 14712, 36, 787, 1178, 40051, 246, 6791, 1437, 475, 17426, 4839, 1246, 9, 41, 20265, 7085, 18140, 1827, 30175, 22703, 36, 17589, 853, 571, 4839, 14, 4548, 999, 9285, 23, 5004, 1389, 479, 50118, 405, 6308, 80, 38898, 118, 8254, 30, 10088, 1437, 46213, 2156, 258, 7501, 30, 2232, 34634, 9, 19790, 22481, 1123, 36, 364, 4, 571, 4, 2156, 1009, 17487, 17487, 17487, 50118, 3226, 25606, 1009, 17487, 17487, 17487, 50118, 3226, 25606, 1009, 17487, 17487, 17487, 50118, 3226, 25606, 1009, 17487, 17487, 17487, 50118, 3226, 25606, 1009, 17487, 17487, 17487, 50118, 35248, 10933, 2485, 9, 2422, 33278, 4791, 23, 10, 731, 9, 508, 39750, 1039, 1178, 40051, 306, 787, 45421, 1459, 4559, 14, 1307, 9883, 9, 2232, 2690, 32, 1455, 19, 41, 14754, 999, 9285, 731, 36, 579, 12997, 4839, 9, 787, 1178, 40051, 245, 1437, 39750, 1039, 1178, 4